# async 与 await

学习目标：能组织串行、并发和有限并发工作，使用异步迭代并处理取消与拒绝。

前置知识：Promise 状态与链式调用、异常处理、迭代器和 ES 模块。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/22-async-await/。

1. [flow.mjs](scripts/22-async-await/flow.mjs)：暂停、拒绝、return await 和清理。
2. [organization.mjs](scripts/22-async-await/organization.mjs)：串行、并发与回调等待。
3. [limited.mjs](scripts/22-async-await/limited.mjs)：有限并发及失败时排空任务。
4. [limited-demo.mjs](scripts/22-async-await/limited-demo.mjs)：并发上限、空输入和失败后等待。
5. [feature.mjs](scripts/22-async-await/feature.mjs)：异步模块初始化及导出。
6. [imports.mjs](scripts/22-async-await/imports.mjs)：两次导入与缺失模块拒绝。
7. [iteration.mjs](scripts/22-async-await/iteration.mjs)：异步协议、生成器与提前关闭。
8. [sync-close.mjs](scripts/22-async-await/sync-close.mjs)：同步生成器产出拒绝值时的关闭与清理暂停。

Step 1：运行暂停与错误传播。

```bash
node scripts/22-async-await/flow.mjs
```

Step 2：运行串行和并发示例。

```bash
node scripts/22-async-await/organization.mjs
```

Step 3：运行有限并发和取消示例。

```bash
node scripts/22-async-await/limited-demo.mjs
```

Step 4：运行动态导入示例。

```bash
node scripts/22-async-await/imports.mjs
```

Step 5：运行异步迭代示例。

```bash
node scripts/22-async-await/iteration.mjs
```

Step 6：比较同步生成器的两种清理路径。

```bash
node scripts/22-async-await/sync-close.mjs
```

## 1 异步函数与 await 的暂停点

async 函数调用总会返回 Promise；普通 return 决定兑现值，未捕获的 throw 决定拒绝原因。函数先执行到第一个 await，再暂停这次函数执行，把控制权交回调用者。await 会采用 Promise 或 thenable 的结果，也接受普通值；即使值已就绪，后续语句仍通过作业恢复，不是同步穿过 await。

try/catch 能捕获所在块内 await 转出的拒绝。若只 return 一个 Promise 而不在 try 中 await，该 Promise 后来拒绝时，函数的 catch 已经离开。finally 仍适合释放资源，但 return 或抛错可能覆盖原本的返回或失败。

配套 [flow.mjs](scripts/22-async-await/flow.mjs)：

```javascript
const events = [];
async function compute() {
  events.push("start");
  const value = await 4;
  events.push("resume");
  return value * 2;
}
const pending = compute();
events.push("caller");
console.log(await pending, events.join(",")); // → 8 start,caller,resume

// 2. return await 在当前 try 内等待，因此拒绝能由本函数捕获。
async function handle() {
  try {
    return await Promise.reject(new Error("read failed"));
  } catch (error) {
    return error.message;
  } finally {
    events.push("cleanup");
  }
}
console.log(await handle(), events.at(-1)); // → read failed cleanup

// 3. 对照：直接返回 Promise 时，下面的 catch 捕获不到其后续拒绝。
async function forward() {
  try {
    return Promise.reject(new Error("forwarded"));
  } catch {
    return "not reached";
  }
}
console.log(await forward().catch((error) => error.message)); // → forwarded
```

## 2 串行、并发与异步回调

两项工作是否必须等前一项结束，取决于调用与 await 的位置，而不是函数有没有 async。

有数据依赖时，在循环内 await；工作相互独立时，先调用函数创建各项工作，再用 Promise.all 等待。并发表示多项工作在时间上重叠，不保证 JavaScript 计算在多个 CPU 上同时运行。

forEach 不使用回调返回值，因此 await array.forEach(async ...) 不会等待这些回调。map 可以收集 Promise，但仍须交给 Promise.all。遗漏 await 还可能让外层过早报告成功或漏接拒绝。下面先展示该观察，再显式等待已启动的工作，避免遗留任务。

![等待的位置决定工作能否重叠。这是调用顺序示意，横向间距不表示真实耗时。](image/illustration/22-01-async-serial-concurrent.svg)

图示说明：依据调用与 await 的先后关系自绘。下排只说明本例的启动与汇合关系，不保证任意任务的完成顺序。

在下面 order 的两组记录中找 start 与 end；再检查 forEach 的返回位置是否已经等待内部工作。

配套 [organization.mjs](scripts/22-async-await/organization.mjs)：

```javascript
const order = [];
async function work(id) {
  order.push(`start${id}`);
  await Promise.resolve();
  order.push(`end${id}`);
  return id * 10;
}
// 每次 await 完成后才进入下一轮，两个任务串行执行。
for (const id of [1, 2]) await work(id);
console.log("serial", order.join(",")); // → serial start1,end1,start2,end2
// 清空观察记录；map 先启动两个任务，all 再等待它们的结果。
order.length = 0;
console.log("values", JSON.stringify(await Promise.all([1, 2].map(work)))); // → values [10,20]
console.log("concurrent", order.join(",")); // → concurrent start1,start2,end1,end2

// 用手动放行的 gate 控制完成时刻，不靠任意延时猜测顺序。
const gate = Promise.withResolvers();
const finished = [];
const pending = [];
await [1, 2].forEach((id) => {
  pending.push((async () => { await gate.promise; finished.push(id); })());
});
console.log("forEach returned", finished.length); // → forEach returned 0
// forEach 已返回；显式放行并等待收集的 Promise，才算全部结束。
gate.resolve();
await Promise.all(pending);
console.log("joined", finished.join(",")); // → joined 1,2
```

## 3 限制并发数量与协作取消

Promise.all 不限制并发数量。下面 mapLimited 接收数组 items、正整数 limit 和处理函数 worker；固定数量的消费者共享 next，每次同步取走一个索引后才 await。results 按输入索引存放，从而与完成顺序解耦。

失败时停止领取新任务，并等待已启动任务结束后抛出首个原因；这是一种明确的本例策略。它不强行终止正在执行的任务。Promise 本身没有取消方法，AbortController 与 AbortSignal 属于宿主 API；只有主动响应 signal 的操作才会取消。取消通常是拒绝，仍须捕获。

配套 [limited.mjs](scripts/22-async-await/limited.mjs)：

```javascript
export async function mapLimited(items, limit, worker) {
  // 本例约定 limit 是正整数，不增加入口检查。
  // 结果按输入下标保存，完成顺序不会打乱返回数组。
  const results = new Array(items.length);
  let next = 0;
  let failed = false;
  let firstError;
  async function consume() {
    while (!failed && next < items.length) {
      // 在 await 前领取唯一索引；多个消费者共享 next，不重复领取。
      const index = next++;
      try {
        results[index] = await worker(items[index]);
      } catch (error) {
        // 记住首个失败并停止领取新任务；已开始的任务仍由各消费者等待。
        if (!failed) { failed = true; firstError = error; }
      }
    }
  }
  // 固定数量的消费者循环取任务；先等待已启动任务结束，再向外抛错。
  await Promise.all(Array.from({ length: Math.min(limit, items.length) }, consume));
  if (failed) throw firstError;
  return results;
}
```

下面用 Node 的 assert 显式比较观察结果：equal 比较值，deepEqual 比较数组等结构，rejects 等待并匹配指定拒绝。符合预期时断言本身不打印，不符合时直接抛出 AssertionError；这些是示例观察条件，不是对教学输入的防御检查。yieldTurn 是 Node 定时器 Promise API 的 setImmediate 别名，让任务在后续事件循环阶段继续。

配套 [limited-demo.mjs](scripts/22-async-await/limited-demo.mjs)：

```javascript
import assert from "node:assert/strict";
import { setImmediate as yieldTurn } from "node:timers/promises";
import { mapLimited } from "./limited.mjs";

// 1. active 记录当前任务数，peak 记录最高值；finally 使失败时也能减回去。
let active = 0;
let peak = 0;
const values = await mapLimited([1, 2, 3, 4, 5], 2, async (value) => {
  active += 1;
  peak = Math.max(peak, active);
  try { await yieldTurn(); return value * 2; }
  finally { active -= 1; }
});
assert.deepEqual(values, [2, 4, 6, 8, 10]);
assert.equal(peak, 2);
assert.equal(active, 0);
assert.deepEqual(await mapLimited([], 2, () => 1), []);

// 2. 首项故意失败，第二项完成，第三项应始终没有被领取。
let drained = false;
const started = [];
await assert.rejects(mapLimited([1, 2, 3], 2, async (value) => {
  started.push(value);
  if (value === 1) throw new Error("worker failed");
  await yieldTurn();
  drained = true;
}), /worker failed/);
assert.deepEqual(started, [1, 2]);
assert.equal(drained, true);
console.log("limit=2 ordered empty failure-drained"); // → limit=2 ordered empty failure-drained

// 3. 取消是另一项宿主能力：先接住拒绝，再发出取消信号。
const controller = new AbortController();
const operation = yieldTurn("unused", { signal: controller.signal });
const rejection = assert.rejects(operation, { name: "AbortError" });
controller.abort();
await rejection;
console.log("cancelled and observed"); // → cancelled and observed
```

## 4 动态导入与顶层 await

import() 是表达式，返回以模块命名空间对象兑现的 Promise；路径解析、加载或求值失败会拒绝。它可以放在条件分支中延迟加载。Node.js 相对模块说明符仍以发起导入的文件为基准，必须写扩展名。

ES 模块允许顶层 await；依赖该异步模块的模块需要等待它完成求值。同一模块 URL 通常只求值一次，重复导入不会重新执行副作用。不要用互相等待的模块初始化制造无法满足的依赖。普通脚本和 CommonJS 顶层不能直接使用 await。

配套 [feature.mjs](scripts/22-async-await/feature.mjs)：

```javascript
console.log("feature evaluated"); // → 对同一模块 URL 的两次导入，仅打印一次
export const factor = await Promise.resolve(3);
export const scale = (value) => value * factor;
```

配套 [imports.mjs](scripts/22-async-await/imports.mjs)：

```javascript
import assert from "node:assert/strict";
const first = await import("./feature.mjs");
const second = await import("./feature.mjs");
assert.equal(first, second);
console.log(first.scale(4)); // → 12
await assert.rejects(import("./not-present.mjs"), { code: "ERR_MODULE_NOT_FOUND" });
console.log("missing module rejected"); // → missing module rejected
```

## 5 异步迭代与提前结束

异步可迭代对象通过 Symbol.asyncIterator 返回异步迭代器，next 返回以 {value, done} 兑现的 Promise。异步生成器用 async function* 声明，可以 await 获取下一项，再 yield 交给消费者。

for await...of 顺序等待每次 next；break 会尝试调用并等待迭代器的 return，因而可触发生成器 finally。它也能把同步迭代器适配为异步迭代器；这不是把所有数据并发读取。

配套 [iteration.mjs](scripts/22-async-await/iteration.mjs)：

```javascript
import assert from "node:assert/strict";
const events = [];
async function* pages() {
  try {
    yield await Promise.resolve("page1");
    yield await Promise.resolve("page2");
  } finally {
    await Promise.resolve();
    events.push("closed");
  }
}
// 先手动调用协议方法：取一页后 return，确认 finally 完成。
const iterator = pages();
assert.equal(iterator[Symbol.asyncIterator](), iterator);
assert.deepEqual(await iterator.next(), { value: "page1", done: false });
await iterator.return();
assert.deepEqual(await iterator.next(), { value: undefined, done: true });
// 再由 for await 管理迭代；break 提前结束，也会请求关闭生成器。
events.length = 0;
for await (const page of pages()) {
  events.push(page);
  break;
}
console.log(events.join(",")); // → page1,closed
// 同步可迭代对象也可进入 for await；其中的 Promise 值会被等待。
const values = [];
for await (const value of [Promise.resolve(2), 3]) values.push(value);
console.log(values.join(",")); // → 2,3
```

ECMAScript 2025 中，同步迭代器适配器等待 next 返回的 value；当 done 为 false 且该值的 Promise 拒绝时，会通过 IteratorClose 尝试调用原迭代器的 return。普通同步生成器因此进入 finally，拒绝仍传给消费者；本章 Node.js 24.11.0 具有这一行为。

关闭尝试不会反复推进生成器。如果 finally 自己再次 yield，后续清理会暂停；下面把 pauseInFinally 设为 true 展示这一边界，并显式 next 完成本例。应避免在必需的清理步骤之间 yield。

配套 [sync-close.mjs](scripts/22-async-await/sync-close.mjs)：

```javascript
import assert from "node:assert/strict";

const events = [];
function* items(pauseInFinally = false) {
  try {
    yield Promise.reject(new Error("item rejected"));
  } finally {
    events.push("cleanup");
    // 专门构造清理过程再次暂停的对照，观察关闭尝试与清理完成的区别。
    if (pauseInFinally) yield "paused";
    events.push("closed");
  }
}
async function consume(iterator) {
  for await (const value of iterator) events.push(value);
}

// 第一组：清理中没有暂停，一次关闭尝试即可执行到 closed。
const source = items();
await assert.rejects(() => consume(source), { name: "Error", message: "item rejected" });
assert.deepEqual(events, ["cleanup", "closed"]);
assert.deepEqual(source.next(), { value: undefined, done: true });
console.log("rejection", events.join(",")); // → rejection cleanup,closed

// 第二组：finally 中再次 yield；随后显式 next 才推进剩余清理。
events.length = 0;
const suspended = items(true);
await assert.rejects(() => consume(suspended), { name: "Error", message: "item rejected" });
assert.deepEqual(events, ["cleanup"]);
console.log("paused", events.join(",")); // → paused cleanup；关闭尝试没有继续推进 finally 中的 yield
assert.deepEqual(suspended.next(), { value: undefined, done: true });
assert.deepEqual(events, ["cleanup", "closed"]);
console.log("resumed", events.join(",")); // → resumed cleanup,closed；显式 next 使本例清理完成
```

## 本章小结

- await 暂停当前异步执行，普通回调 API 是否等待 Promise 要分别判断。
- 控制任务创建、限制并发、收集失败和响应取消是不同职责。
- 异步模块与迭代器需要明确完成条件及提前结束时的清理。

## 练习

1. 只修改 limited-demo.mjs 第一组正常任务，分别将上限设为 1 和 3；每次重置 active、peak，并同步修改峰值断言，其他故障与取消示例保持原上限 2。记录峰值；标准：结果顺序不变，峰值不超过上限，结束时 active 为 0。
2. 只在 iteration.mjs 的 for await 循环去掉 break，保留遍历前 events.length = 0；标准：取得两页且只执行一次 finally。
3. 只修改 limited-demo.mjs 第二组故障场景：保持上限 2，让第二项经过 yieldTurn 后也抛错，保留 drained 标记赋值并区分两条错误消息；标准：所有已启动工作结束后才报告失败，不能出现未处理拒绝。

### 提示

1. limit 只改变消费者数量；同一输入有五项工作。
2. finally 在生成器自然结束时也执行。
3. 保留第一项立即失败，第二项稍后失败，以明确首个原因。


### 参考解析

1. 第一组给定任务各有一次异步等待，上限 1/3 分别观察峰值 1/3；返回数组仍为 [2,4,6,8,10]，active 归零。同步更新 assert.equal(peak, limit)，不要把仍检查 2 的断言误认成算法失败。
2. 事件变为 page1,page2,closed，两个 yield 都消费完，finally 仍只执行一次。
3. 第一项使 failed 设为 true，第三项不再领取；第二项完成自己的 finally 或收尾后抛错，consume 捕获它但保留已记录的首个原因。Promise.all 等两个消费者都结束后，mapLimited 才抛出第一项的 worker failed。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TC39（ECMA-262 第 16 版） | [§27.7.5 Async Functions](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-async-functions-abstract-operations)、同页 §27.1.1.3–4 异步协议、[§27.1.6.4 AsyncFromSyncIteratorContinuation](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-asyncfromsynciteratorcontinuation) 的拒绝关闭、同页 §27.6 异步生成器及 [§27.5.3.6 GeneratorYield](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-generatoryield) 的暂停；[§7.4.11 IteratorClose](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-iteratorclose)；[§14.7.5 for-in/of](https://tc39.es/ecma262/2025/multipage/ecmascript-language-statements-and-declarations.html#sec-for-in-and-for-of-statements)与 [§13.3.10 Import Calls](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-import-calls)。 |
| Node.js 24.11.0 | [ESM 的 import()、URL 缓存和顶层 await](https://nodejs.org/download/release/v24.11.0/docs/api/esm.html)、[AbortController](https://nodejs.org/download/release/v24.11.0/docs/api/globals.html#class-abortcontroller)、[Timers Promises API](https://nodejs.org/download/release/v24.11.0/docs/api/timers.html#timers-promises-api)、[assert.rejects](https://nodejs.org/download/release/v24.11.0/docs/api/assert.html#assertrejectsasyncfn-error-message)：宿主调度、取消和示例断言。 |
